# Faruq-v3 — CAFR-YOLO seed42 causal ablation (Colab)

Notebook development untuk menjalankan urutan **C1 → C2 → C3 → C4 → CAFR** pada Faruq-v3 grouped development.

**Kontrak eksperimen**
- YOLO26n native tetap sebagai detector.
- Semua arm mulai dari checkpoint D0 seed42 yang sama.
- Seed dikunci **42**.
- Evaluasi hanya pada `val`.
- Split `test` tidak boleh tersedia di `DATA_ROOT`.
- Hasil/checkpoint ditulis langsung ke Google Drive dan runner dapat resume dari `last.pt`.
- CAFR final mengkalibrasi patch size hanya dari label `train`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import importlib, json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path
import torch

assert torch.cuda.is_available(), 'Aktifkan GPU: Runtime > Change runtime type > T4 GPU.'

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/cafr-yolo'

os.chdir('/content')
if REPO.exists():
    shutil.rmtree(REPO)

clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH,
         'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(clone)
    if result.returncode == 0:
        break
    if REPO.exists():
        shutil.rmtree(REPO)
    if attempt == 3:
        raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)

for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)

sys.path.insert(0, str(REPO / 'src'))
importlib.invalidate_caches()
os.chdir(REPO)

import ultralytics
print('GPU        :', torch.cuda.get_device_name(0))
print('Ultralytics:', ultralytics.__version__)
assert ultralytics.__version__ == '8.4.96'
print('Branch     :', BRANCH)


In [ ]:
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
))

ARCHIVE = require_project_artifact(PROJECT_ROOT, 'bundles/faruq-development-v3-grouped.tar')
D0_CHECKPOINT = require_project_artifact(
    PROJECT_ROOT,
    'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
)

DATA_ROOT = Path('/content/faruq-development-v3-grouped')
GROUPED_SUMMARY = DATA_ROOT / 'faruq_grouped_summary.json'
OUTPUT_ROOT = PROJECT_ROOT / 'experiments/faruq-v3-cafr-seed42-v1'

if not GROUPED_SUMMARY.is_file():
    if DATA_ROOT.exists():
        shutil.rmtree(DATA_ROOT)
    with tarfile.open(ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')

assert (DATA_ROOT / 'data.yaml').is_file(), DATA_ROOT
assert GROUPED_SUMMARY.is_file(), GROUPED_SUMMARY
assert not (DATA_ROOT / 'test').exists(), 'STOP: test tidak boleh tersedia pada development CAFR.'
assert D0_CHECKPOINT.is_file(), D0_CHECKPOINT

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT   :', PROJECT_ROOT)
print('ARCHIVE        :', ARCHIVE)
print('DATA_ROOT      :', DATA_ROOT)
print('GROUPED_SUMMARY:', GROUPED_SUMMARY)
print('D0_CHECKPOINT  :', D0_CHECKPOINT)
print('OUTPUT_ROOT    :', OUTPUT_ROOT)


## Pilih arm

Secara default notebook menjalankan semua arm seed42 secara berurutan:

- `C1`: luminance-guided shared RGB gate
- `C2`: C1 + radial × directional decomposition
- `C3`: C2 + soft entropy-conditioned selection
- `C4`: C3 + unsigned 180° orientation representation
- `CAFR`: C4 + patch calibration dari skala bbox training

Jika runtime terputus, jalankan ulang notebook. Runner akan menggunakan checkpoint `last.pt` bila kontraknya cocok.


In [ ]:
RUN_ARMS = ['C1', 'C2', 'C3', 'C4', 'CAFR']
print('RUN_ARMS:', RUN_ARMS)


In [ ]:
def run_arm(arm: str):
    command = [
        sys.executable, '-u', '-m',
        'coffee_detector.experiments.run_faruq_v3_cafr_arm',
        '--arm', arm,
        '--data-root', str(DATA_ROOT),
        '--grouped-summary', str(GROUPED_SUMMARY),
        '--d0-checkpoint', str(D0_CHECKPOINT),
        '--output-root', str(OUTPUT_ROOT),
        '--seed', '42',
        '--device', '0',
        '--latency-iterations', '50',
        '--authorize-training',
    ]
    print('\n' + '=' * 100)
    print('MENJALANKAN:', ' '.join(command), flush=True)
    print('=' * 100)

    log_dir = OUTPUT_ROOT / 'logs'
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / f'{arm}_seed42.log'

    with log_path.open('a', encoding='utf-8') as log:
        process = subprocess.Popen(
            command,
            cwd=REPO,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in process.stdout:
            log.write(line)
            log.flush()
            if (
                line.startswith(('START ', 'RESUME ', 'CAFR PATCH CALIBRATION:', 'SPECTRAL', 'CAFR'))
                or 'mAP50-95' in line
                or 'Epoch' in line
            ):
                print(line, end='', flush=True)
        return_code = process.wait()

    if return_code != 0:
        tail = ''.join(log_path.read_text(errors='replace').splitlines(keepends=True)[-160:])
        print(tail)
        raise RuntimeError(f'{arm} gagal dengan exit code {return_code}. Lihat {log_path}')

    result_path = OUTPUT_ROOT / 'val_reports' / f'{arm}_seed42_result.json'
    assert result_path.is_file(), result_path
    result = json.loads(result_path.read_text(encoding='utf-8'))
    assert result['evaluation_split'] == 'val'
    assert result['test_images_accessed'] is False
    print(f'DONE {arm}:', result_path)
    return result

RESULTS = {}
for arm in RUN_ARMS:
    RESULTS[arm] = run_arm(arm)

print('\nSEMUA ARM YANG DIPILIH SELESAI.')


## Ringkasan headline

Tabel ini **deskriptif untuk seed42 development validation**. Jangan buka locked test dari notebook ini.


In [ ]:
import pandas as pd
from IPython.display import display

rows = []
for arm in RUN_ARMS:
    result_path = OUTPUT_ROOT / 'val_reports' / f'{arm}_seed42_result.json'
    if not result_path.is_file():
        continue
    result = json.loads(result_path.read_text(encoding='utf-8'))
    m = result['metrics']
    rows.append({
        'arm': arm,
        'macro_mAP50_95': m.get('macro_map50_95'),
        'bottom3_mAP50_95': m.get('bottom3_class_map50_95'),
        'worst_mAP50_95': m.get('worst_class_map50_95'),
        'mAP50_95': m.get('metrics/mAP50-95(B)'),
        'mAP50': m.get('metrics/mAP50(B)'),
        'precision': m.get('metrics/precision(B)'),
        'recall': m.get('metrics/recall(B)'),
        'latency_median_ms': result.get('latency', {}).get('median_ms'),
        'latency_p95_ms': result.get('latency', {}).get('p95_ms'),
        'patch_size': result.get('cafr', {}).get('patch_size'),
        'test_accessed': result.get('test_images_accessed'),
    })

df = pd.DataFrame(rows)
display(df)

ranked = df.sort_values(
    ['macro_mAP50_95', 'bottom3_mAP50_95', 'worst_mAP50_95'],
    ascending=False,
)
print('\nRANKING DESKRIPTIF SEED42:')
display(ranked[['arm','macro_mAP50_95','bottom3_mAP50_95','worst_mAP50_95',
                'latency_median_ms','patch_size']])

calibration_path = OUTPUT_ROOT / 'val_reports/cafr_patch_calibration.json'
if calibration_path.is_file():
    print('\nCAFR PATCH CALIBRATION:')
    print(calibration_path.read_text(encoding='utf-8'))

print('\nOUTPUT:', OUTPUT_ROOT)
print('Kirim tabel ini ke chat. Jangan jalankan test.')


## Catatan keputusan

Setelah seed42 selesai, **jangan otomatis melatih seed123/2026**. Kita baca dulu:

1. Apakah C1 membantu → shared luminance/chromaticity-preserving gate?
2. Apakah C2 menambah manfaat → radial information?
3. Apakah C3 membantu → soft selection?
4. Apakah C4 membantu → unsigned orientation symmetry?
5. Apakah patch calibration CAFR mempertahankan/meningkatkan hasil?

Baru setelah itu maksimal dua kandidat dipromosikan ke multi-seed.
